# Model 2: BERTopic

This notebook applies BERTopic to the preprocessed 20 Newsgroups dataset. BERTopic is a modern topic modeling technique that uses transformer-based sentence embeddings, UMAP for dimensionality reduction, and HDBSCAN for clustering — then extracts topic representations using a class-based TF-IDF procedure.

**Input**: `s3://topic-modeling-demo/02_preprocessing/newsgroups_preprocessed.parquet`  
**Output**: Visualizations saved to `./output/`, model saved to `./output/bertopic_model.pkl`

## Imports

In [1]:
import subprocess, sys

def install_if_missing(str_package, str_pip_name=None, lst_extra_args=None):
    """Install a package if it is not already available."""
    try:
        __import__(str_package)
    except ImportError:
        lst_cmd = [sys.executable, '-m', 'pip', 'install', '-q']
        if lst_extra_args:
            lst_cmd.extend(lst_extra_args)
        lst_cmd.append(str_pip_name or str_package)
        subprocess.check_call(lst_cmd)

# ensure numpy <2 is installed first (compatible with GCC 7.x on SageMaker)
install_if_missing('numpy', 'numpy<2')

# ensure cython and setuptools are present for hdbscan build
install_if_missing('Cython', 'cython')
install_if_missing('setuptools')

# install hdbscan with --no-build-isolation so it uses the numpy already installed
# (otherwise pip's build isolation fetches numpy 2.4+ which needs GCC >= 9.3)
install_if_missing('hdbscan', 'hdbscan', lst_extra_args=['--no-build-isolation'])

install_if_missing('umap', 'umap-learn')
install_if_missing('sentence_transformers', 'sentence-transformers')
install_if_missing('bertopic')
install_if_missing('wordcloud')
install_if_missing('optuna')

import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud
from gensim.models import CoherenceModel
from gensim import corpora

print('All imports successful')

All imports successful


## Helper Class

In [ ]:
class BERTopicModel:
    """BERTopic topic modeling with transformer embeddings."""
    
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3', region_name='us-east-1')
        self.df_data = None
        self.topic_model = None
        self.lst_topics = None
        self.arr_probs = None
        self.arr_embeddings = None
        self.arr_reduced_embeddings = None
        self.set_plot_style()
    
    def set_plot_style(self):
        """Configure matplotlib/seaborn styling."""
        sns.set_style('whitegrid')
        plt.rcParams['figure.figsize'] = (14, 6)
        plt.rcParams['font.size'] = 11
    
    def import_data(self):
        """Load preprocessed data from S3."""
        str_uri = f's3://{self.str_bucket}/02_preprocessing/newsgroups_preprocessed.parquet'
        self.df_data = pd.read_parquet(str_uri)
        print(f'Loaded {len(self.df_data):,} posts from S3')
    
    def generate_embeddings(self):
        """Generate sentence embeddings using a pretrained transformer."""
        print('\nGenerating sentence embeddings (this may take a few minutes)...')
        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.arr_embeddings = embedding_model.encode(
            self.df_data['text_clean'].tolist(),
            show_progress_bar=True,
            batch_size=64
        )
        print(f'Embeddings shape: {self.arr_embeddings.shape}')
    
    def _build_and_evaluate(self, int_min_cluster_size, int_min_samples,
                            int_n_neighbors, int_n_components):
        """Build a BERTopic model with given params and return coherence."""
        umap_model = UMAP(
            n_neighbors=int_n_neighbors,
            n_components=int_n_components,
            min_dist=0.0,
            metric='cosine',
            random_state=42
        )
        hdbscan_model = HDBSCAN(
            min_cluster_size=int_min_cluster_size,
            min_samples=int_min_samples,
            metric='euclidean',
            prediction_data=True
        )
        vectorizer_model = CountVectorizer(
            stop_words='english',
            ngram_range=(1, 2),
            min_df=10
        )
        model = BERTopic(
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            top_n_words=10,
            verbose=False
        )
        
        try:
            lst_topics, _ = model.fit_transform(
                self.df_data['text_clean'].tolist(),
                embeddings=self.arr_embeddings
            )
        except ValueError:
            # happens when min_cluster_size is too large, producing too few
            # topics for the CountVectorizer min_df setting
            return -1.0
        
        # compute coherence
        lst_topic_ids = [int_t for int_t in model.get_topic_info()['Topic'] if int_t != -1]
        if len(lst_topic_ids) < 2:
            return -1.0  # not enough topics
        
        lst_topic_words = []
        for int_t in lst_topic_ids[:20]:
            lst_words = [str_w for str_w, _ in model.get_topic(int_t)[:10]]
            lst_topic_words.append(lst_words)
        
        lst_texts = [str_doc.split() for str_doc in self.df_data['text_clean']]
        dictionary = corpora.Dictionary(lst_texts)
        dictionary.filter_extremes(no_below=15, no_above=0.5)
        
        coherence_model = CoherenceModel(
            topics=lst_topic_words,
            texts=lst_texts,
            dictionary=dictionary,
            coherence='c_v'
        )
        return coherence_model.get_coherence()
    
    def tune_hyperparameters(self, int_n_trials=10):
        """Use Optuna to tune BERTopic hyperparameters."""
        print(f'\nTuning hyperparameters with {int_n_trials} Optuna trials...')
        pbar = tqdm(total=int_n_trials, desc='Optuna tuning')
        
        def objective(trial):
            int_min_cluster_size = trial.suggest_int('min_cluster_size', 50, 300, step=25)
            int_min_samples = trial.suggest_int('min_samples', 5, 25, step=5)
            int_n_neighbors = trial.suggest_int('n_neighbors', 10, 30, step=5)
            int_n_components = trial.suggest_int('n_components', 3, 10)
            
            flt_coherence = self._build_and_evaluate(
                int_min_cluster_size, int_min_samples,
                int_n_neighbors, int_n_components
            )
            pbar.update(1)
            pbar.set_postfix({'coherence': f'{flt_coherence:.4f}'})
            return flt_coherence
        
        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=int_n_trials)
        pbar.close()
        
        print(f'\nBest trial:')
        print(f'  Coherence: {study.best_trial.value:.4f}')
        print(f'  Params: {study.best_trial.params}')
        
        return study.best_trial.params
    
    def fit_model(self, dict_params=None, int_min_topic_size=150):
        """Fit BERTopic model with tuned or default parameters."""
        if dict_params:
            int_min_cluster_size = dict_params['min_cluster_size']
            int_min_samples = dict_params['min_samples']
            int_n_neighbors = dict_params['n_neighbors']
            int_n_components = dict_params['n_components']
        else:
            int_min_cluster_size = int_min_topic_size
            int_min_samples = 10
            int_n_neighbors = 15
            int_n_components = 5
        
        print(f'\nFitting BERTopic (min_cluster_size={int_min_cluster_size}, '
              f'min_samples={int_min_samples}, n_neighbors={int_n_neighbors}, '
              f'n_components={int_n_components})...')
        
        umap_model = UMAP(
            n_neighbors=int_n_neighbors,
            n_components=int_n_components,
            min_dist=0.0,
            metric='cosine',
            random_state=42
        )
        hdbscan_model = HDBSCAN(
            min_cluster_size=int_min_cluster_size,
            min_samples=int_min_samples,
            metric='euclidean',
            prediction_data=True
        )
        vectorizer_model = CountVectorizer(
            stop_words='english',
            ngram_range=(1, 2),
            min_df=10
        )
        
        self.topic_model = BERTopic(
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            top_n_words=10,
            verbose=True
        )
        
        self.lst_topics, self.arr_probs = self.topic_model.fit_transform(
            self.df_data['text_clean'].tolist(),
            embeddings=self.arr_embeddings
        )
        
        # get reduced embeddings for visualization
        umap_2d = UMAP(n_components=2, min_dist=0.0, metric='cosine', random_state=42)
        self.arr_reduced_embeddings = umap_2d.fit_transform(self.arr_embeddings)
        
        int_n_topics = len(set(self.lst_topics)) - (1 if -1 in self.lst_topics else 0)
        int_outliers = self.lst_topics.count(-1) if isinstance(self.lst_topics, list) else (np.array(self.lst_topics) == -1).sum()
        print(f'\nTopics discovered: {int_n_topics}')
        print(f'Outlier documents: {int_outliers:,} ({int_outliers/len(self.lst_topics)*100:.1f}%)')
    
    def plot_top_words_per_topic(self):
        """Plot top words for each BERTopic topic."""
        df_topic_info = self.topic_model.get_topic_info()
        df_topics = df_topic_info[df_topic_info['Topic'] != -1].head(20)
        
        int_n_topics = len(df_topics)
        int_cols = 4
        int_rows = (int_n_topics + int_cols - 1) // int_cols
        
        fig, axes = plt.subplots(int_rows, int_cols, figsize=(20, 4 * int_rows))
        axes = axes.flatten()
        
        for int_i, int_topic_id in enumerate(df_topics['Topic'].values):
            lst_topic_words = self.topic_model.get_topic(int_topic_id)
            if not lst_topic_words:
                continue
            lst_words = [str_w for str_w, _ in lst_topic_words[:10]]
            lst_scores = [flt_s for _, flt_s in lst_topic_words[:10]]
            
            axes[int_i].barh(range(len(lst_words)), lst_scores,
                             color='#DD8452', edgecolor='black')
            axes[int_i].set_yticks(range(len(lst_words)))
            axes[int_i].set_yticklabels(lst_words)
            axes[int_i].invert_yaxis()
            axes[int_i].set_title(f'Topic {int_topic_id} ({df_topics[df_topics["Topic"]==int_topic_id]["Count"].values[0]:,} docs)',
                                  fontsize=11, fontweight='bold')
            axes[int_i].set_xlabel('c-TF-IDF Score', fontsize=9)
        
        for int_j in range(int_i + 1, len(axes)):
            axes[int_j].set_visible(False)
        
        plt.suptitle('Top Words per BERTopic Topic', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_top_words_per_topic.png',
                    bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 01_top_words_per_topic.png')
    
    def plot_word_clouds(self):
        """Generate word clouds for BERTopic topics."""
        df_topic_info = self.topic_model.get_topic_info()
        df_topics = df_topic_info[df_topic_info['Topic'] != -1].head(20)
        
        int_n_topics = len(df_topics)
        int_cols = 4
        int_rows = (int_n_topics + int_cols - 1) // int_cols
        
        fig, axes = plt.subplots(int_rows, int_cols, figsize=(20, 4 * int_rows))
        axes = axes.flatten()
        
        for int_i, int_topic_id in enumerate(df_topics['Topic'].values):
            lst_topic_words = self.topic_model.get_topic(int_topic_id)
            if not lst_topic_words:
                continue
            dict_word_scores = {str_w: max(flt_s, 0.001) for str_w, flt_s in lst_topic_words[:30]}
            
            wordcloud = WordCloud(
                width=400, height=300,
                background_color='white',
                colormap='copper',
                max_words=30
            ).generate_from_frequencies(dict_word_scores)
            
            axes[int_i].imshow(wordcloud, interpolation='bilinear')
            axes[int_i].set_title(f'Topic {int_topic_id}', fontsize=12, fontweight='bold')
            axes[int_i].axis('off')
        
        for int_j in range(int_i + 1, len(axes)):
            axes[int_j].set_visible(False)
        
        plt.suptitle('Word Clouds per BERTopic Topic', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_word_clouds.png',
                    bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 02_word_clouds.png')
    
    def plot_topic_distribution(self):
        """Plot distribution of documents across topics."""
        df_topic_info = self.topic_model.get_topic_info()
        df_topics = df_topic_info[df_topic_info['Topic'] != -1].head(20).sort_values('Count', ascending=True)
        
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.barh(range(len(df_topics)), df_topics['Count'].values,
                color='#DD8452', edgecolor='black')
        ax.set_yticks(range(len(df_topics)))
        ax.set_yticklabels([f'Topic {int_t}' for int_t in df_topics['Topic'].values])
        ax.set_xlabel('Number of Documents', fontsize=12, fontweight='bold')
        ax.set_title('Document Count by BERTopic Topic', fontsize=14, fontweight='bold')
        
        for int_i, int_v in enumerate(df_topics['Count'].values):
            ax.text(int_v + max(df_topics['Count'].values)*0.01, int_i, f'{int_v:,}',
                    ha='left', va='center', fontsize=9)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_topic_distribution.png',
                    bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 03_topic_distribution.png')
    
    def plot_document_clusters(self):
        """Plot 2D UMAP projection of documents colored by topic."""
        fig, ax = plt.subplots(figsize=(14, 10))
        
        arr_topics = np.array(self.lst_topics)
        
        # plot outliers first (gray)
        mask_outlier = arr_topics == -1
        if mask_outlier.sum() > 0:
            ax.scatter(self.arr_reduced_embeddings[mask_outlier, 0],
                       self.arr_reduced_embeddings[mask_outlier, 1],
                       c='lightgray', s=5, alpha=0.3, label='Outlier')
        
        # plot topic documents
        lst_unique_topics = sorted(set(arr_topics[~mask_outlier]))
        cmap = plt.cm.get_cmap('tab20', len(lst_unique_topics))
        
        for int_i, int_topic in enumerate(lst_unique_topics[:20]):
            mask = arr_topics == int_topic
            ax.scatter(self.arr_reduced_embeddings[mask, 0],
                       self.arr_reduced_embeddings[mask, 1],
                       c=[cmap(int_i)], s=8, alpha=0.5, label=f'Topic {int_topic}')
        
        ax.set_xlabel('UMAP Dimension 1', fontsize=12, fontweight='bold')
        ax.set_ylabel('UMAP Dimension 2', fontsize=12, fontweight='bold')
        ax.set_title('Document Clusters (UMAP Projection)', fontsize=14, fontweight='bold')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, markerscale=3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/04_document_clusters.png',
                    bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 04_document_clusters.png')
    
    def plot_topic_heatmap(self):
        """Plot topic similarity heatmap."""
        df_topic_info = self.topic_model.get_topic_info()
        lst_topic_ids = df_topic_info[df_topic_info['Topic'] != -1]['Topic'].head(20).tolist()
        int_n_topics = len(lst_topic_ids)
        
        # build topic-word matrix for similarity
        set_all_words = set()
        for int_t in lst_topic_ids:
            lst_words = self.topic_model.get_topic(int_t)
            if lst_words:
                for str_w, _ in lst_words[:10]:
                    set_all_words.add(str_w)
        
        lst_all_words = sorted(set_all_words)
        arr_matrix = np.zeros((int_n_topics, len(lst_all_words)))
        
        for int_i, int_t in enumerate(lst_topic_ids):
            dict_topic = dict(self.topic_model.get_topic(int_t)[:10])
            for int_j, str_w in enumerate(lst_all_words):
                arr_matrix[int_i, int_j] = dict_topic.get(str_w, 0)
        
        # compute cosine similarity
        from sklearn.metrics.pairwise import cosine_similarity
        arr_sim = cosine_similarity(arr_matrix)
        
        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(arr_sim, xticklabels=[f'T{int_t}' for int_t in lst_topic_ids],
                    yticklabels=[f'T{int_t}' for int_t in lst_topic_ids],
                    cmap='YlOrRd', ax=ax, annot=True, fmt='.2f',
                    linewidths=0.5, vmin=0, vmax=1)
        ax.set_title('Topic Similarity Heatmap (BERTopic)', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/05_topic_similarity.png',
                    bbox_inches='tight', dpi=150)
        plt.show()
        print('Saved: 05_topic_similarity.png')
    
    def compute_coherence(self):
        """Compute c_v coherence for comparison with LDA."""
        lst_topic_ids = [int_t for int_t in self.topic_model.get_topic_info()['Topic'] if int_t != -1]
        lst_topic_words = []
        for int_t in lst_topic_ids[:20]:
            lst_words = [str_w for str_w, _ in self.topic_model.get_topic(int_t)[:10]]
            lst_topic_words.append(lst_words)
        
        lst_texts = [str_doc.split() for str_doc in self.df_data['text_clean']]
        dictionary = corpora.Dictionary(lst_texts)
        dictionary.filter_extremes(no_below=15, no_above=0.5)
        
        coherence_model = CoherenceModel(
            topics=lst_topic_words,
            texts=lst_texts,
            dictionary=dictionary,
            coherence='c_v'
        )
        flt_coherence = coherence_model.get_coherence()
        print(f'BERTopic Coherence (c_v): {flt_coherence:.4f}')
        return flt_coherence
    
    def save_model(self):
        """Save the BERTopic model locally."""
        dict_model_data = {
            'topic_model': self.topic_model,
            'lst_topics': self.lst_topics,
            'arr_reduced_embeddings': self.arr_reduced_embeddings
        }
        str_model_path = f'{self.str_dirname_output}/bertopic_model.pkl'
        with open(str_model_path, 'wb') as f:
            pickle.dump(dict_model_data, f)
        print(f'Model saved to {str_model_path}')
    
    def print_topics(self):
        """Print all topics with their top words."""
        df_topic_info = self.topic_model.get_topic_info()
        df_topics = df_topic_info[df_topic_info['Topic'] != -1]
        
        print('\n' + '='*60)
        print('BERTOPIC TOPICS')
        print('='*60)
        for _, row in df_topics.head(20).iterrows():
            int_topic_id = row['Topic']
            lst_words = self.topic_model.get_topic(int_topic_id)
            if lst_words:
                str_words = ', '.join([str_w for str_w, _ in lst_words[:10]])
                print(f'\nTopic {int_topic_id} ({row["Count"]:,} docs): {str_words}')
        print('\n' + '='*60)

## Constants

In [3]:
# S3 configuration
str_bucket = 'topic-modeling-demo'

# output directory
str_dirname_output = './output'

# tuning settings
int_n_trials = 10

print(f'S3 Bucket: {str_bucket}')
print(f'Output Directory: {str_dirname_output}')
print(f'Optuna Trials: {int_n_trials}')

S3 Bucket: topic-modeling-demo
Output Directory: ./output
Optuna Trials: 10


## Output Directory

In [4]:
try:
    os.mkdir(str_dirname_output)
    print(f'Created output directory: {str_dirname_output}')
except FileExistsError:
    print(f'Output directory already exists: {str_dirname_output}')

Created output directory: ./output


## Load Data and Generate Embeddings

In [5]:
cls_bertopic = BERTopicModel(str_bucket=str_bucket, str_dirname_output=str_dirname_output)
cls_bertopic.import_data()

Loaded 17,841 posts from S3


In [6]:
cls_bertopic.generate_embeddings()


Generating sentence embeddings (this may take a few minutes)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/279 [00:00<?, ?it/s]

Embeddings shape: (17841, 384)


## Tune Hyperparameters

In [7]:
dict_best_params = cls_bertopic.tune_hyperparameters(int_n_trials=int_n_trials)


Tuning hyperparameters with 10 Optuna trials...


Optuna tuning:  50%|█████     | 5/10 [04:34<04:16, 51.28s/it, coherence=0.6712][W 2026-03-23 17:47:53,214] Trial 5 failed with parameters: {'min_cluster_size': 275, 'min_samples': 5, 'n_neighbors': 25, 'n_components': 4} because of the following error: ValueError('max_df corresponds to < documents than min_df').
Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_8838/2590653704.py", line 105, in objective
    flt_coherence = self._build_and_evaluate(
                    ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_8838/2590653704.py", line 67, in _build_and_evaluate
    lst_topics, _ = model.fit_transform(
                    ^^^^^^^^^^^^^^^^^^^^
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/bertopic/_bertopic.py", line 520, in fit_transform
    self._ext

ValueError: max_df corresponds to < documents than min_df

## Fit Final Model

In [ ]:
cls_bertopic.fit_model(dict_params=dict_best_params)

## Visualize Results

In [ ]:
cls_bertopic.plot_top_words_per_topic()

In [ ]:
cls_bertopic.plot_word_clouds()

In [ ]:
cls_bertopic.plot_topic_distribution()

In [ ]:
cls_bertopic.plot_document_clusters()

In [ ]:
cls_bertopic.plot_topic_heatmap()

## Compute Coherence and Save Model

In [ ]:
flt_coherence = cls_bertopic.compute_coherence()

In [ ]:
cls_bertopic.save_model()

In [ ]:
cls_bertopic.print_topics()